# Walmart Sales Forecasting — Temporal Fusion Transformer

**მოდელი:** Temporal Fusion Transformer (TFT)
**კატეგორია:** Deep Learning (LSTM encoder/decoder + multi-head attention + gating)
**ბიბლიოთეკა:** `neuralforecast` (Nixtla)
**Logging:** WandB (project: `walmart-forecasting`)

**Runs:**
1. `TFT_Baseline` — temporal-only input, small hidden size
2. `TFT_ExogEnriched` — static (Store, Dept) + future-known (IsHoliday, calendar) covariates added
3. `TFT_Tuned` — full-year context, larger hidden size and deeper LSTM
4. `TFT_Final` — best configuration, trained on the full dataset


## 1. Setup

In [ ]:
# ინსტალაცია
!pip install neuralforecast wandb --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from neuralforecast.losses.pytorch import MAE

import wandb
from pytorch_lightning.loggers import WandbLogger

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Drive mount
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/walmart'
DATA_DIR = f'{PROJECT_DIR}/data'
MODELS_DIR = f'{PROJECT_DIR}/models'

import os
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Data: {DATA_DIR}")
print(f"Models: {MODELS_DIR}")

In [ ]:
from google.colab import userdata

try:
    wandb_key = userdata.get('WANDB_API_KEY')
    wandb.login(key=wandb_key)
    print("WandB logged in")
except Exception as e:
    wandb.login()

WANDB_PROJECT = "walmart-forecasting"
print(f"WandB project: {WANDB_PROJECT}")

## 2. მონაცემების ჩატვირთვა და მომზადება

TFT-ს, ისევე როგორც ნებისმიერ `neuralforecast` მოდელს, სჭირდება **long-format** მონაცემები:
- `unique_id` — ცალკეული time series (Store × Dept კომბინაცია)
- `ds` — datestamp
- `y` — target (Weekly_Sales)

დამატებით, TFT-ის მთავარი უპირატესობა ჩვეულებრივ sequence-to-sequence მოდელებთან შედარებით არის სამი ტიპის input-ის ერთდროული გამოყენება:
- **static** — დროში უცვლელი მახასიათებლები თითო სერიაზე (Store, Dept)
- **future-known (futr_exog)** — მახასიათებლები, რომლებიც წინასწარაა ცნობილი ჰორიზონტზეც (calendar-based: IsHoliday, კვირის ნომრის sin/cos ტრანსფორმაცია)
- **historic-only (hist_exog)** — მხოლოდ წარსულში ცნობილი სიგნალები

რადგან ჩვენ ვიყენებთ მხოლოდ `train.csv.zip` / `test.csv.zip`-ს (features.csv/stores.csv არ გვაქვს), historic-only exogenous სიგნალები ცალკე არ გვაქვს დამატებული — `y`-ის თავად ავტორეგრესიული history ამ როლს ასრულებს. `IsHoliday` ორივე ფაილში (train და test) არსებობს, ასე რომ ის უსაფრთხოდ შეიძლება გამოვიყენოთ როგორც **future-known** — ის calendar-based ცნობადია წინასწარ, leakage არ არის.

In [ ]:
# Raw მონაცემები
train_raw = pd.read_csv(f'{DATA_DIR}/train.csv.zip')
test_raw = pd.read_csv(f'{DATA_DIR}/test.csv.zip')

train_raw['Date'] = pd.to_datetime(train_raw['Date'])
test_raw['Date'] = pd.to_datetime(test_raw['Date'])

print(f"Train: {train_raw.shape}")
print(f"Test:  {test_raw.shape}")
print(f"Unique stores: {train_raw['Store'].nunique()}")
print(f"Unique depts:  {train_raw['Dept'].nunique()}")
print(f"Unique (Store, Dept) combos: {train_raw.groupby(['Store', 'Dept']).ngroups}")

In [ ]:
# Calendar-based future-known features — ცნობილია ნებისმიერ თარიღზე წინასწარ
def add_calendar_features(df):
    df = df.copy()
    week = df['Date'].dt.isocalendar().week.astype(float)
    df['weekofyear_sin'] = np.sin(2 * np.pi * week / 52.0)
    df['weekofyear_cos'] = np.cos(2 * np.pi * week / 52.0)
    df['IsHoliday'] = df['IsHoliday'].astype(float)
    return df


train_raw = add_calendar_features(train_raw)
test_raw = add_calendar_features(test_raw)

FUTR_EXOG = ['IsHoliday', 'weekofyear_sin', 'weekofyear_cos']
STAT_EXOG = ['Store', 'Dept']


def to_long_format(df, has_target=True):
    """Neuralforecast-ის required long-format-ში გადავყავართ, exogenous სვეტების შენარჩუნებით."""
    df = df.copy()
    df['unique_id'] = df['Store'].astype(str) + '_' + df['Dept'].astype(str)
    df = df.rename(columns={'Date': 'ds'})
    keep_cols = ['unique_id', 'ds'] + FUTR_EXOG + STAT_EXOG
    if has_target:
        df = df.rename(columns={'Weekly_Sales': 'y'})
        keep_cols = ['unique_id', 'ds', 'y'] + FUTR_EXOG + STAT_EXOG
    return df[keep_cols]


train_nf = to_long_format(train_raw, has_target=True)
test_nf = to_long_format(test_raw, has_target=False)

# Static exogenous ცხრილი — თითო unique_id-ზე ერთი მწკრივი (Store/Dept დროში არ იცვლება)
static_df = (
    train_nf[['unique_id'] + STAT_EXOG]
    .drop_duplicates('unique_id')
    .reset_index(drop=True)
)
static_df[STAT_EXOG] = static_df[STAT_EXOG].astype(float)

# Temporal df-ში static სვეტები აღარ გვჭირდება (ცალკე static_df-ით მიეწოდება მოდელს)
train_nf = train_nf.drop(columns=STAT_EXOG)
test_nf = test_nf.drop(columns=STAT_EXOG)

print(f"Train nf: {train_nf.shape}")
print(f"Test nf:  {test_nf.shape}")
print(f"Static df: {static_df.shape}")
print(f"\nSample:")
train_nf.head()

In [ ]:
# Time series length distribution — რამდენად გრძელი time series-ები გვაქვს?
series_lengths = train_nf.groupby('unique_id').size()
print(f"Time series count: {len(series_lengths)}")
print(f"Length stats:")
print(f"  Min:    {series_lengths.min()}")
print(f"  Median: {series_lengths.median():.0f}")
print(f"  Max:    {series_lengths.max()}")
print(f"  Mean:   {series_lengths.mean():.1f}")

# ძალიან მოკლე time series-ები TFT-ის LSTM encoder-ს/attention-ს ეშლება, ვფილტრავთ
MIN_LENGTH = 52  # ერთი წელი მინიმუმ
valid_ids = series_lengths[series_lengths >= MIN_LENGTH].index
train_nf = train_nf[train_nf['unique_id'].isin(valid_ids)].reset_index(drop=True)
static_df = static_df[static_df['unique_id'].isin(valid_ids)].reset_index(drop=True)

print(f"\nAfter filtering (min length {MIN_LENGTH}):")
print(f"  Time series: {train_nf['unique_id'].nunique()}")
print(f"  Total rows:  {len(train_nf)}")

## 3. Train/Val Split + WMAE Metric

In [ ]:
def wmae(y_true, y_pred, weights):
    """Competition metric."""
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)


def evaluate_forecast(forecasts_df, val_df, train_raw_val, pred_col='TFT'):
    """
    TFT-ის forecasts-ს ვადარებთ ვალიდაციის ground truth-ს.
    Weights ვიღებთ train_raw_val-იდან (IsHoliday-ის მიხედვით), 5x holiday weighting მხოლოდ ევალუაციაზე.
    """
    # მხოლოდ y-ს ვიღებთ val_df-იდან — val_df-ს უკვე აქვს თავისი futr_exog სვეტები
    # (IsHoliday, weekofyear_sin/cos), რომლებიც დუბლირებულ სვეტს გამოიწვევდა merge-ზე
    merged = forecasts_df.merge(val_df[['unique_id', 'ds', 'y']], on=['unique_id', 'ds'], how='inner')

    # IsHoliday-ს ვუბრუნდებით raw train-იდან
    train_raw_val_lookup = train_raw_val.copy()
    train_raw_val_lookup['unique_id'] = (
        train_raw_val_lookup['Store'].astype(str) + '_' + train_raw_val_lookup['Dept'].astype(str)
    )
    train_raw_val_lookup = train_raw_val_lookup.rename(columns={'Date': 'ds'})

    merged = merged.merge(
        train_raw_val_lookup[['unique_id', 'ds', 'IsHoliday']],
        on=['unique_id', 'ds'], how='left'
    )

    weights = np.where(merged['IsHoliday'] == True, 5, 1)
    return wmae(merged['y'].values, merged[pred_col].values, weights)

In [ ]:
# Time-based split — ბოლო 12 კვირა ვალიდაცია
train_nf_sorted = train_nf.sort_values(['unique_id', 'ds']).reset_index(drop=True)

VAL_HORIZON = 12  # ბოლო 12 კვირა val-ისთვის

# Per-series split
def per_series_split(df, val_h):
    train_parts, val_parts = [], []
    for uid, group in df.groupby('unique_id'):
        group = group.sort_values('ds')
        if len(group) > val_h:
            train_parts.append(group.iloc[:-val_h])
            val_parts.append(group.iloc[-val_h:])
    return pd.concat(train_parts).reset_index(drop=True), pd.concat(val_parts).reset_index(drop=True)


train_split, val_split = per_series_split(train_nf_sorted, VAL_HORIZON)

# Raw train ვინახოთ val period-ისთვის (weights-ისთვის)
val_dates = val_split['ds'].unique()
train_raw_val_period = train_raw[train_raw['Date'].isin(val_dates)]

print(f"Train split: {train_split.shape}, series: {train_split['unique_id'].nunique()}")
print(f"Val split:   {val_split.shape}, series: {val_split['unique_id'].nunique()}")
print(f"Val date range: {val_split['ds'].min()} → {val_split['ds'].max()}")

## 4. Run 1 — `TFT_Baseline`

Temporal-only კონფიგურაცია, static/future-known covariates-ის გარეშე — მხოლოდ ავტორეგრესიული history, პირდაპირ შედარებადი ბაზისური წერტილი TFT-ის architecture-ის (LSTM + attention) წვლილის შესაფასებლად:
- Input size = 26 კვირა
- Horizon = 12 კვირა (val period)
- hidden_size = 64, n_head = 4 (Nixtla-ს დოკუმენტაციასა და TFT ლიტერატურაში orte ორივეგან სტანდარტული default არჩევანია `n_head=4`; `hidden_size` T4-ის ბიუჯეტისთვის მცირედ დაწეულია 128 default-იდან)
- lstm_layers = 1 (darts/TFT დოკუმენტაცია: "1 is a good default")
- dropout = 0.1
- 300 training steps

In [ ]:
H = VAL_HORIZON  # forecast horizon = 12

BASELINE_CONFIG = {
    'h': H,
    'input_size': 26,
    'max_steps': 300,
    'learning_rate': 1e-3,
    'batch_size': 32,
    'hidden_size': 64,
    'n_head': 4,
    'n_rnn_layers': 1,
    'dropout': 0.1,
}

# WandB run
wandb.finish() if wandb.run else None

run = wandb.init(
    project=WANDB_PROJECT,
    name="TFT_Baseline",
    config=BASELINE_CONFIG,
    reinit=True,
    tags=['tft', 'baseline']
)

wandb_logger = WandbLogger(project=WANDB_PROJECT, name="TFT_Baseline", experiment=run)

model_baseline = TFT(
    h=H,
    input_size=BASELINE_CONFIG['input_size'],
    loss=MAE(),
    max_steps=BASELINE_CONFIG['max_steps'],
    learning_rate=BASELINE_CONFIG['learning_rate'],
    batch_size=BASELINE_CONFIG['batch_size'],
    hidden_size=BASELINE_CONFIG['hidden_size'],
    n_head=BASELINE_CONFIG['n_head'],
    n_rnn_layers=BASELINE_CONFIG['n_rnn_layers'],
    dropout=BASELINE_CONFIG['dropout'],
    scaler_type='robust',
    random_seed=42,
    logger=wandb_logger,
    enable_progress_bar=True,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
)

nf_baseline = NeuralForecast(models=[model_baseline], freq='W-FRI')
nf_baseline.fit(df=train_split, val_size=H)

# Prediction on val
forecasts_baseline = nf_baseline.predict()
val_wmae_baseline = evaluate_forecast(forecasts_baseline, val_split, train_raw_val_period)

wandb.log({'val_wmae': val_wmae_baseline})
wandb.summary['val_wmae'] = val_wmae_baseline
print(f"\nBaseline Val WMAE: {val_wmae_baseline:.2f}")

wandb.finish()

## 5. Run 2 — `TFT_ExogEnriched`

TFT-ის მთავარი უპირატესობა heterogeneous input-ების ერთდროული გამოყენებაა. ამ run-ში ვამატებთ:
- **Static exogenous** — `Store`, `Dept` (variable selection network თითო სერიისთვის სპეციფიკურ წონებს ისწავლის)
- **Future-known exogenous** — `IsHoliday`, `weekofyear_sin/cos` (calendar-based, leakage-ის გარეშე ცნობილი მთელ ჰორიზონტზე)

ეს განსაკუთრებით მნიშვნელოვანია Walmart-ის დატასეტისთვის — holiday flag პირდაპირ მიემართება 5x-წონიან ევალუაციის კვირებს, ასე რომ მოდელს წინასწარვე შეუძლია მათი ამოცნობა, ნაცვლად history-დან მარტო-implicit სწავლისა.

In [ ]:
EXOG_CONFIG = {
    'h': H,
    'input_size': 26,
    'max_steps': 400,
    'learning_rate': 1e-3,
    'batch_size': 32,
    'hidden_size': 64,
    'n_head': 4,
    'n_rnn_layers': 1,
    'dropout': 0.1,
}

wandb.finish() if wandb.run else None

run = wandb.init(
    project=WANDB_PROJECT,
    name="TFT_ExogEnriched",
    config=EXOG_CONFIG,
    reinit=True,
    tags=['tft', 'exog_enriched']
)

wandb_logger = WandbLogger(project=WANDB_PROJECT, name="TFT_ExogEnriched", experiment=run)

model_exog = TFT(
    h=H,
    input_size=EXOG_CONFIG['input_size'],
    loss=MAE(),
    max_steps=EXOG_CONFIG['max_steps'],
    learning_rate=EXOG_CONFIG['learning_rate'],
    batch_size=EXOG_CONFIG['batch_size'],
    hidden_size=EXOG_CONFIG['hidden_size'],
    n_head=EXOG_CONFIG['n_head'],
    n_rnn_layers=EXOG_CONFIG['n_rnn_layers'],
    dropout=EXOG_CONFIG['dropout'],
    futr_exog_list=FUTR_EXOG,
    stat_exog_list=STAT_EXOG,
    scaler_type='robust',
    random_seed=42,
    logger=wandb_logger,
    enable_progress_bar=True,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
)

nf_exog = NeuralForecast(models=[model_exog], freq='W-FRI')
nf_exog.fit(df=train_split, static_df=static_df, val_size=H)

# predict() ჰორიზონტი train_split-ის მიღმაა (ანუ ზუსტად val_split-ის თარიღები),
# ამიტომ future-known covariates იმ თარიღებზე ცალკე უნდა მივაწოდოთ
val_futr_df = val_split[['unique_id', 'ds'] + FUTR_EXOG]
forecasts_exog = nf_exog.predict(futr_df=val_futr_df)
val_wmae_exog = evaluate_forecast(forecasts_exog, val_split, train_raw_val_period)

wandb.log({'val_wmae': val_wmae_exog})
wandb.summary['val_wmae'] = val_wmae_exog
print(f"\nExogEnriched Val WMAE: {val_wmae_exog:.2f}")

wandb.finish()

## 6. Run 3 — `TFT_Tuned`

ორი მიმართულებით ვზრდით მოდელს:
1. **Input window 52 კვირაზე** — მთელი წლის კონტექსტი, რომ Thanksgiving/Christmas/Super Bowl-ის წლიური სეზონურობა ჩანდეს (იგივე მიზეზი, რაც ხანგრძლივი context window-ის საჭიროებას ნებისმიერ sequence მოდელში წარმოშობს ამ დატასეტზე).
2. **hidden_size 128-მდე, n_rnn_layers 2-მდე, dropout 0.2-მდე** — Nixtla-ს default hidden_size სწორედ 128-ია, ხოლო TFT-ზე ჩატარებულ hyperparameter search კვლევებში (მაგ. market-forecasting გრიდში) 2-კომპონენტიანი LSTM stack და 0.15–0.25 dropout რეგულარულად სჯობნის 1-ფენიან/დაბალი dropout-იან ვარიანტებს ხმაურიან, holiday-სპაიკებიან სერიებზე.

batch_size 64-მდე გავზარდე — attention-ის მქონე მოდელს დიდი batch უფრო სტაბილურ გრადიენტს აძლევს, T4-ის მეხსიერებაც საკმარისია ამ hidden_size-ზე.

In [ ]:
TUNED_CONFIG = {
    'h': H,
    'input_size': 52,
    'max_steps': 600,
    'learning_rate': 5e-4,
    'batch_size': 64,
    'hidden_size': 128,
    'n_head': 4,
    'n_rnn_layers': 2,
    'dropout': 0.2,
}

wandb.finish() if wandb.run else None

run = wandb.init(
    project=WANDB_PROJECT,
    name="TFT_Tuned",
    config=TUNED_CONFIG,
    reinit=True,
    tags=['tft', 'tuned']
)

wandb_logger = WandbLogger(project=WANDB_PROJECT, name="TFT_Tuned", experiment=run)

model_tuned = TFT(
    h=H,
    input_size=TUNED_CONFIG['input_size'],
    loss=MAE(),
    max_steps=TUNED_CONFIG['max_steps'],
    learning_rate=TUNED_CONFIG['learning_rate'],
    batch_size=TUNED_CONFIG['batch_size'],
    hidden_size=TUNED_CONFIG['hidden_size'],
    n_head=TUNED_CONFIG['n_head'],
    n_rnn_layers=TUNED_CONFIG['n_rnn_layers'],
    dropout=TUNED_CONFIG['dropout'],
    futr_exog_list=FUTR_EXOG,
    stat_exog_list=STAT_EXOG,
    scaler_type='robust',
    random_seed=42,
    logger=wandb_logger,
    enable_progress_bar=True,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
)

# გავფილტროთ ის series-ები რომლებიც input_size+H სიგრძეზე ნაკლებია
tuned_min_length = TUNED_CONFIG['input_size'] + H
tuned_valid_ids = train_split.groupby('unique_id').size()
tuned_valid_ids = tuned_valid_ids[tuned_valid_ids >= tuned_min_length].index
train_split_tuned = train_split[train_split['unique_id'].isin(tuned_valid_ids)].reset_index(drop=True)
val_split_tuned = val_split[val_split['unique_id'].isin(tuned_valid_ids)].reset_index(drop=True)
static_df_tuned = static_df[static_df['unique_id'].isin(tuned_valid_ids)].reset_index(drop=True)

print(f"Series after long-context filter: {train_split_tuned['unique_id'].nunique()}")

nf_tuned = NeuralForecast(models=[model_tuned], freq='W-FRI')
nf_tuned.fit(df=train_split_tuned, static_df=static_df_tuned, val_size=H)

val_futr_df_tuned = val_split_tuned[['unique_id', 'ds'] + FUTR_EXOG]
forecasts_tuned = nf_tuned.predict(futr_df=val_futr_df_tuned)
val_wmae_tuned = evaluate_forecast(forecasts_tuned, val_split_tuned, train_raw_val_period)

wandb.log({'val_wmae': val_wmae_tuned})
wandb.summary['val_wmae'] = val_wmae_tuned
print(f"\nTuned Val WMAE: {val_wmae_tuned:.2f}")

wandb.finish()

## 7. Run 4 — `TFT_Final`

საუკეთესო კონფიგურაცია, გატრენინგებული **მთელ train set-ზე** (train + val). ეს არის მოდელი რომელიც test-ზე ვიწინასწარმეტყველებთ და submissions-ს ვქმნით.

In [ ]:
# რომელი conf იყო საუკეთესო?
run_results = {
    'baseline': val_wmae_baseline,
    'exog_enriched': val_wmae_exog,
    'tuned': val_wmae_tuned,
}
best_run = min(run_results, key=run_results.get)
print(f"Best run so far: {best_run} (WMAE={run_results[best_run]:.2f})")

BEST_CONFIG = {
    'baseline': BASELINE_CONFIG,
    'exog_enriched': EXOG_CONFIG,
    'tuned': TUNED_CONFIG,
}[best_run]

FINAL_CONFIG = {**BEST_CONFIG, 'max_steps': BEST_CONFIG['max_steps'] * 2}  # უფრო მეტი training
print(f"Final config: {FINAL_CONFIG}")

# საჭიროა თუ არა exogenous, დამოკიდებულია საუკეთესო run-ზე
FINAL_USE_EXOG = best_run != 'baseline'

In [ ]:
wandb.finish() if wandb.run else None

run = wandb.init(
    project=WANDB_PROJECT,
    name="TFT_Final",
    config=FINAL_CONFIG,
    reinit=True,
    tags=['tft', 'final']
)

wandb_logger = WandbLogger(project=WANDB_PROJECT, name="TFT_Final", experiment=run)

model_final = TFT(
    h=H,
    input_size=FINAL_CONFIG['input_size'],
    loss=MAE(),
    max_steps=FINAL_CONFIG['max_steps'],
    learning_rate=FINAL_CONFIG['learning_rate'],
    batch_size=FINAL_CONFIG['batch_size'],
    hidden_size=FINAL_CONFIG['hidden_size'],
    n_head=FINAL_CONFIG['n_head'],
    n_rnn_layers=FINAL_CONFIG['n_rnn_layers'],
    dropout=FINAL_CONFIG['dropout'],
    futr_exog_list=FUTR_EXOG if FINAL_USE_EXOG else None,
    stat_exog_list=STAT_EXOG if FINAL_USE_EXOG else None,
    scaler_type='robust',
    random_seed=42,
    logger=wandb_logger,
    enable_progress_bar=True,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
)

# Filter series to match input_size requirement
final_min_length = FINAL_CONFIG['input_size'] + H
final_valid_ids = train_nf_sorted.groupby('unique_id').size()
final_valid_ids = final_valid_ids[final_valid_ids >= final_min_length].index
train_full_filtered = train_nf_sorted[train_nf_sorted['unique_id'].isin(final_valid_ids)].reset_index(drop=True)
static_df_final = static_df[static_df['unique_id'].isin(final_valid_ids)].reset_index(drop=True)

print(f"Training on full data: {train_full_filtered.shape}, series: {train_full_filtered['unique_id'].nunique()}")

nf_final = NeuralForecast(models=[model_final], freq='W-FRI')
nf_final.fit(df=train_full_filtered, static_df=static_df_final if FINAL_USE_EXOG else None)

# Test set-ისთვის future-known covariates საჭიროა predict()-ში, რადგან ეს თარიღები
# train_full_filtered-ში არ გვხვდება
test_futr_df = test_nf[test_nf['unique_id'].isin(final_valid_ids)].reset_index(drop=True)

if FINAL_USE_EXOG:
    forecasts_final = nf_final.predict(futr_df=test_futr_df)
else:
    forecasts_final = nf_final.predict()

print(f"\nFinal forecasts shape: {forecasts_final.shape}")
print(forecasts_final.head())

wandb.summary['best_val_wmae_from_experiments'] = run_results[best_run]
wandb.summary['config_used'] = best_run
wandb.finish()

## 8. Model შენახვა

TFT მოდელი შენახვისთვის:
1. **Drive-ზე** — pickle backup
2. **WandB Artifact** — versioned storage WandB-ში (Model Registry ანალოგი)

MLflow Model Registry-ში მოგვიანებით დავარეგისტრირებთ — ინფერენს notebook-ში.

In [ ]:
import pickle

# Drive-ზე შენახვა
tft_save_path = f'{MODELS_DIR}/tft_final.pkl'
with open(tft_save_path, 'wb') as f:
    pickle.dump(nf_final, f)

# Also save the forecasts
forecasts_final.to_csv(f'{MODELS_DIR}/tft_forecasts.csv', index=False)

print(f"Model saved: {tft_save_path}")
print(f"Forecasts saved: {MODELS_DIR}/tft_forecasts.csv")

In [ ]:
# WandB Artifact-ად ატვირთვა (Model Registry-ის ანალოგი)
wandb.finish() if wandb.run else None
run = wandb.init(project=WANDB_PROJECT, name="TFT_ModelArtifact", reinit=True)

artifact = wandb.Artifact(name="tft_final", type="model")
artifact.add_file(tft_save_path)
artifact.add_file(f'{MODELS_DIR}/tft_forecasts.csv')

run.log_artifact(artifact)
print("Model uploaded to WandB Artifacts")

wandb.finish()

## 9. პროგნოზების შემოწმება

In [ ]:
# რამდენიმე ცხრილი — რას პროგნოზირებს მოდელი
print(f"Test predictions: {len(forecasts_final)} rows")
print(f"Unique series: {forecasts_final['unique_id'].nunique()}")
print(f"\nSample predictions:")
sample_ids = forecasts_final['unique_id'].unique()[:3]
for uid in sample_ids:
    sub = forecasts_final[forecasts_final['unique_id'] == uid].head(3)
    print(f"\n=== {uid} ===")
    print(sub.to_string(index=False))

In [ ]:
# ვიზუალიზაცია — რამდენიმე time series-ის actual vs predicted
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for ax, uid in zip(axes, sample_ids[:3]):
    # Historical
    hist = train_full_filtered[train_full_filtered['unique_id'] == uid].tail(52)
    # Forecast
    fc = forecasts_final[forecasts_final['unique_id'] == uid]

    ax.plot(hist['ds'], hist['y'], label='Historical', color='steelblue')
    ax.plot(fc['ds'], fc['TFT'], label='TFT Forecast', color='orange', linestyle='--')
    ax.set_title(f'Series: {uid}')
    ax.legend()

plt.tight_layout()
plt.show()

## 10. შეჯამება

TFT-ის სამი კონფიგურაცია გავცადეთ WandB-ზე ლოგირებით:

- **Baseline** — temporal-only, 26-კვირიანი input, hidden_size=64
- **ExogEnriched** — Store/Dept static covariates + IsHoliday/calendar future-known covariates
- **Tuned** — 52-კვირიანი (მთელი წლის) input, hidden_size=128, 2-ფენიანი LSTM encoder/decoder

საუკეთესო კონფიგურაცია ავტომატურად შეირჩა ვალიდაციის WMAE-ის მიხედვით (იხ. Run 4-ის უჯრედი) და გადატრენინგდა 2x steps-ით მთელ ტრეინინგ სეტზე. შენახულია Drive-ზე pickle-ად და WandB Artifact-ად.

TFT-ის N-BEATS-ისგან განსხვავებული ძირითადი უპირატესობა — heterogeneous input-ების (static + future-known + observed history) ერთდროული, gated/attention-based შერწყმა, რაც განსაკუთრებით მნიშვნელოვანია holiday-სპაიკებიან, სტორი-სპეციფიკურ სერიებზე, სადაც წინასწარ ცნობილი კალენდარული სიგნალი პირდაპირ ეხმარება მოდელს.

XGBoost-თან და N-BEATS-თან პირდაპირი შედარება inference notebook-ში იქნება, Kaggle submission-ის ქულის მიხედვით.